In [ ]:
import pandas as pd
import string
import math
import re 
from tqdm.auto import tqdm
from scipy.stats import kendalltau
from sklearn.model_selection import train_test_split
seed = 14

df = pd.read_csv("./data_set.csv", sep="\t")


train, test = train_test_split(df, test_size=0.2, random_state=seed)
train, valid = train_test_split(train, test_size=0.1, random_state=seed)

valid.reset_index(inplace=True)
valid.drop(columns=["index"], inplace=True)



In [ ]:
train.shape, valid.shape, 

((377, 4), (42, 4), (105, 4))

In [3]:
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize, word_tokenize  
def tokenize_text(text):
    tokens = [word for word in word_tokenize(text, language='portuguese')]
    return len(tokens)


df["len_academico"] = df["estilo_academico"].apply(lambda x: tokenize_text(x))
df["len_autor"] = df["estilo_autoral"].apply(lambda x: tokenize_text(x))

[nltk_data] Downloading package punkt to /home/pablo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
df["len_academico"].sum(), df["len_autor"].sum()

(15603, 11066)

In [5]:
df["estilo_autoral"]

0       @ @ Se Lula e o PT colocarem as mãos não cabe...
1       @ As esquerdas adoram os pobres, os miserávei...
2       @ Os ministros do STF, rasgando a constituiçã...
3       @ @ @ Os ministros do STF, rasgando a constit...
4       @ Conseguiram roubar do contribuinte com a ch...
                             ...                        
519     JOHNNY BRAVO VIAJA DE CAMINHÃO NA BR 163!!!.....
520     Vamos orar e pedir para que o presidente Bols...
521     OS COMUNISTAS INFILTRADOS ENTRE OS CONSERVADO...
522     Bolsonaro manda outra Banana para a imprensa ...
523                         Carnaval dos Bichos 0 via @ 
Name: estilo_autoral, Length: 524, dtype: object

In [6]:
import os
base_dir_i  = "./prompt_v1/"
base_dir_ii = "./prompt_v2/"
predicts_i = []
predicts_ii = []
for ele in os.listdir(base_dir_i):
    if ele.endswith(".txt"):
        predicts_i.append(base_dir_i+ele)

for ele in os.listdir(base_dir_ii):
    if ele.endswith(".txt"):
        predicts_ii.append(base_dir_ii+ele)

In [7]:
predicts_ii

['./prompt_v2/preds_deepseek.txt',
 './prompt_v2/preds_gemma.txt',
 './prompt_v2/preds_llama.txt',
 './prompt_v2/preds_qwen.txt']

In [8]:

models_i = []
for i in range(0, len(predicts_i)):
    df = pd.read_csv(predicts_i[i], sep="\t", header=None)
    df["model"] = predicts_i[i].split("/")[-1].split(".")[0]
    df["prompt_version"] = "v1"
    models_i.append(df.rename(columns={0: "model_outs"}, inplace=False))
    
models_ii = []
for i in range(0, len(predicts_ii)):
    df = pd.read_csv(predicts_ii[i], sep="\t", header=None)
    df["model"] = predicts_ii[i].split("/")[-1].split(".")[0]
    df["prompt_version"] = "v2"
    models_ii.append(df.rename(columns={0: "model_outs"}, inplace=False))

In [9]:
def normalize_answer(s):
    """Lower text and remove punctuation and extra whitespace."""


    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_punc(lower(s)))

import evaluate
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
from scipy.stats import gmean
import numpy

def calculate_BERTSCORE(predictions, references):

  bleurt = evaluate.load("bertscore")
  return numpy.mean(bleurt.compute(predictions=predictions, references=references, lang="pt")["f1"])



#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)



def compute_labse(candidates, references):

    distances = []
    tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/LaBSE')
    model = AutoModel.from_pretrained('sentence-transformers/LaBSE')

    # Tokenize sentences
    tokens_candidates = tokenizer(candidates, padding=True, truncation=True, return_tensors='pt')
    tokens_references = tokenizer(references, padding=True, truncation=True, return_tensors='pt')
    # Compute token embeddings
    with torch.no_grad():
        model_candidates = model(**tokens_candidates)
        model_references = model(**tokens_references)

    # Perform pooling
    embeddings_candidates = F.normalize(mean_pooling(model_candidates, tokens_candidates['attention_mask']), p=2, dim=1)
    embeddings_references = F.normalize(mean_pooling(model_references, tokens_references['attention_mask']), p=2, dim=1)

    for ele1, ele2 in zip(embeddings_candidates, embeddings_references):
        ele1 = ele1.unsqueeze(0)
        ele2 = ele2.unsqueeze(0)
        distances.append(F.cosine_similarity(ele1, ele2))
    return numpy.mean(distances)


def compute_acc(candidates, true_labels):

    labels = {'r2_lu_876': 0,
 'r2_cl_2165': 1,
 'r2_gl_1071': 2,
 'r2_bo_344': 3,
 'r2_bo_208': 4,
 'r2_lu_94': 5}
    model     = AutoModelForSequenceClassification.from_pretrained("../dataset/data/ustance/classification/results/checkpoint-520/")
    tokenizer = AutoTokenizer.from_pretrained("pablocosta/bertabaporu-base-uncased")
    tokenizer.padding_side = "right"  # Fix weird overflow issue with fp16 training
    model = TextClassificationPipeline(model=model, tokenizer=tokenizer, top_k=1)
    preds = []
    for pred in model(candidates, truncation=True, padding=True, max_length=512):
        preds.append(int(pred[0]["label"].split("LABEL_")[1]))
    results = f1_score(preds, [labels[x] for x in true_labels], average="weighted")

    return results


In [10]:
concat_models_i =[]
concat_models_ii =[]

for df in models_i:
    df.reset_index(inplace=True)
    df.drop(columns=["index"], inplace=True)
    concat_models_i.append(pd.concat([df, valid], axis=1))

for df in models_ii:
    df.reset_index(inplace=True)
    df.drop(columns=["index"], inplace=True)
    concat_models_ii.append(pd.concat([df, valid], axis=1))

In [11]:
"""calculate the metrics and format in to a pandas dataframe"""

model_name = []
f1_scores = []
labse_scores = []
acc_scores = []
gmean_scores = []

for df in concat_models_i:
    df["model_clean"] = df["model_outs"].apply(lambda x: normalize_answer(x))
    df["original_clean"] = df["estilo_autoral"].apply(lambda x: normalize_answer(x))

    fscore = calculate_BERTSCORE(df["model_clean"].tolist(), df["original_clean"].tolist())
    labse  = compute_labse(df["model_clean"].tolist(), df["original_clean"].tolist())
    acc    = compute_acc(df["model_clean"].tolist(), valid["user_id"].tolist())
    gmean_score = gmean([fscore, labse, acc])
    model_name.append(df['model'][0]+"v1")
    f1_scores.append(fscore)
    labse_scores.append(labse)
    acc_scores.append(acc)
    gmean_scores.append(gmean_score)



Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [ ]:
valid["user_id"].value_counts()

user_id
r2_bo_344     11
r2_gl_1071    11
r2_cl_2165     6
r2_lu_876      5
r2_lu_94       5
r2_bo_208      4
Name: count, dtype: int64

In [12]:
for df in concat_models_ii:
    df["model_clean"] = df["model_outs"].apply(lambda x: normalize_answer(x))
    df["original_clean"] = df["estilo_autoral"].apply(lambda x: normalize_answer(x))

    fscore = calculate_BERTSCORE(df["model_clean"].tolist(), df["original_clean"].tolist())
    labse  = compute_labse(df["model_clean"].tolist(), df["original_clean"].tolist())
    acc    = compute_acc(df["model_clean"].tolist(), valid["user_id"].tolist())
    gmean_score = gmean([fscore, labse, acc])
    model_name.append(df['model'][0]+"v2")
    f1_scores.append(fscore)
    labse_scores.append(labse)
    acc_scores.append(acc)
    gmean_scores.append(gmean_score)

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [13]:
import random
def get_naive_classifier(source_samples, target_samples, p):
    preds = []
    for i in range(len(source_samples)):
        if random.random() < p:
            preds.append(source_samples[i])
        else:
            preds.append(random.choice(target_samples))
    return preds

naive_generated = get_naive_classifier(valid["estilo_academico"],valid["estilo_autoral"], p=0.5)

In [14]:
fscore = calculate_BERTSCORE(naive_generated, df["original_clean"].tolist())
labse  = compute_labse(naive_generated, df["original_clean"].tolist())
acc    = compute_acc(naive_generated, valid["user_id"].tolist())
gmean_score = gmean([fscore, labse, acc])
model_name.append("naive")
f1_scores.append(fscore)
labse_scores.append(labse)
acc_scores.append(acc)
gmean_scores.append(gmean_score)

Device set to use cuda:0


In [15]:
copy = valid["estilo_academico"].tolist()

fscore = calculate_BERTSCORE(copy, df["original_clean"].tolist())
labse  = compute_labse(copy, df["original_clean"].tolist())
acc    = compute_acc(copy, valid["user_id"].tolist())
gmean_score = gmean([fscore, labse, acc])
model_name.append("copy")
f1_scores.append(fscore)
labse_scores.append(labse)
acc_scores.append(acc)
gmean_scores.append(gmean_score)

Device set to use cuda:0


In [16]:
pd.DataFrame.from_dict({
    "model": model_name,
    "f1_score": f1_scores,
    "labse_score": labse_scores,
    "acc_score": acc_scores,
    "gmean_score": gmean_scores})

,model,f1_score,labse_score,acc_score,gmean_score
0,preds_deepseekv1,0.782531,0.821812,0.476609,0.674236
1,preds_gemmav1,0.707400,0.744868,0.392205,0.591225
2,preds_llamav1,0.761042,0.793815,0.505202,0.673283
3,preds_qwenv1,0.767224,0.807591,0.418519,0.637690
4,preds_deepseekv2,0.772259,0.813415,0.471065,0.666375
5,preds_gemmav2,0.704855,0.736735,0.269474,0.519170
6,preds_llamav2,0.742204,0.775125,0.478739,0.650626
7,preds_qwenv2,0.773682,0.818870,0.539471,0.699169
8,naive,0.664951,0.675865,0.448040,0.586123
9,copy,0.692765,0.751756,0.495301,0.636567
